# 1105 45일차

## 1. 데이터셋 리스트

데이터셋 여러 개를 리스트에 담아 같은 코드로 차례로 돌리는 것 (`m06_all_estimators1_datasets`, `m06_all_estimators2_datasets`)

데이터마다 파일을 따로 두면 모델 목록이나 평가 방식을 고칠 때 전부 손봐야 함. 리스트로 묶으면 바깥 for문만 한 번 돌면 됨

1. `datasets = [iris, cancer, wine, digits]` : `return_X_y=True`로 받은 튜플을 리스트에 담음
2. `data_name = ['iris', 'cancer', 'wine', 'digits']` : 출력에 쓸 이름을 같은 순서로 둠
3. `for index, value in enumerate(datasets):` : 번호와 데이터를 같이 받음
4. `x, y = value` : 튜플을 풀어 `x`, `y`로 나눔
5. `np.argmax(model_acc)` : 점수가 가장 높은 자리의 번호

- `enumerate`의 번호로 `data_name[index]`를 찍어서 어느 데이터의 결과인지 구분함
- 모델 이름과 점수를 각각 리스트에 모아 두고, 마지막에 `np.array`로 바꿔 `argmax`로 최고 모델만 뽑음
- `train_test_split`과 스케일링은 바깥 for문 안에 둠. 데이터마다 나누고 맞춰야 다른 데이터의 분포가 섞이지 않음
- 안쪽 for문이 `all_estimators` 목록이라 데이터 하나마다 모델 수십 개가 돌아감
- 분류는 `StratifiedKFold`, 회귀는 `KFold`로 나눔. 회귀는 y가 연속값이라 클래스 비율을 맞출 수 없음

## 2. 파라미터 튜닝

파라미터 값을 바꿔 가며 여러 번 돌려 가장 좋은 조합을 찾는 것 (`m07_parameter_tuning_using_for`)

손으로 값을 고쳐 돌리면 조합 수만큼 실행해야 하고 점수도 따로 적어 둬야 함. for문에 넣으면 한 번 실행으로 전부 돌고 최고 조합만 남음

1. `learning_rate = [0.1, 0.05, 0.01, 0.005, 0.001]`, `max_depth = [3, 4, 5, 6, 7]` : 후보 값을 리스트로 둠
2. 이중 for문 : 두 리스트의 모든 조합을 만듦
3. `if score > best_score:` : 더 높은 점수가 나오면 점수와 조합을 갱신

- 리스트 길이를 곱한 만큼 모델이 훈련됨. 5 × 5면 25번
- 모델을 만드는 줄이 for문 안에 있어야 조합마다 새 모델이 생김
- `>`로 비교하면 점수가 같을 때 먼저 나온 조합이 남음
- `max_depth`는 트리 깊이. 깊으면 과적합, 얕으면 학습 부족
- **`model.score(x_test, y_test)`로 조합을 고르면 test를 보고 고른 셈이라 그 점수를 성능으로 쓸 수 없음**

## 3. GridSearchCV

파라미터 조합마다 교차 검증을 돌려 최적 조합을 찾아 주는 sklearn 기능 (`m08_GridSearchCV_01_digits`, `m08_GridSearchCV_02_diabetes`)

for문으로 하면 조합을 만들고 점수를 갱신하는 코드를 직접 써야 하고 fold 나누기도 따로 해야 함. 모델을 감싸 두면 조합 × fold를 전부 돌리고 최적 조합으로 다시 훈련까지 함

1. `parameters = [{...}, {...}, {...}]` : 딕셔너리 목록. 키는 파라미터 이름, 값은 후보 리스트
2. `GridSearchCV(xgb, parameters, cv=kfold, verbose=1, n_jobs=1)` : 모델을 감싼 모델
3. `model.fit(x_train, y_train)` : 조합마다 교차 검증을 돌림
4. `model.best_params_` : 가장 좋았던 파라미터 조합
5. `model.best_estimator_` : 그 조합으로 만든 모델
6. `model.best_score_` : 그 조합의 교차 검증 평균 점수

- 딕셔너리를 여러 개 두면 각 딕셔너리 안에서만 조합을 만듦. 같이 쓸 일이 없는 파라미터를 따로 묶어 둘 수 있음
- 한 딕셔너리의 조합 수는 값 리스트 길이의 곱. 조합 수를 다 더한 값에 fold 수를 곱한 만큼 훈련함
- 거기에 한 번을 더 훈련함. 최적 조합으로 train 전체를 다시 학습해서 `best_estimator_`를 만드는 몫
- `verbose=1`은 조합 수와 진행 상황을 찍음
- `n_jobs`는 동시에 쓸 코어 수. 1이면 한 개만 씀
- `model.score(x_test, y_test)`는 `best_estimator_`로 test를 평가함
- `best_estimator_`를 찍으면 내가 준 파라미터 외에 기본값까지 다 나옴. 고른 값만 보려면 `best_params_`

#### cf) best_score_와 model.score

- `best_score_`는 train을 나눠 만든 검증 점수. 조합을 train 안에서 골랐으므로 test 점수보다 높게 나옴
- **성능으로 적을 값은 `model.score(x_test, y_test)` 쪽.** 조합을 고르는 데 쓰지 않은 데이터라서
- 두 값의 차이가 벌어지면 그 조합이 train에만 맞춰진 것